# Corazón de Pulpa · Actuación real con LivePortrait (imagen + video del celular)

LivePortrait copia en Don Harold los gestos de una persona: boca, ojos, cejas y cabeza. Es la opción más expresiva, y además la voz puede ser la de ustedes.

**Cómo grabar cada línea** (una por video, con el nombre de su fila: `1.20.mp4`, `1.07.mp4`):
- Celular **en horizontal** y quieto (apoyado), cara **de frente**, a unos 50 cm, con buena luz de frente y fondo liso.
- Empiecen **un segundo con la cara neutra y la boca cerrada**; después digan la línea **exagerando como en telenovela**.
- No tapen la cara con las manos y giren la cabeza solo un poco.

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar** y luego **Ejecutar todo**.
2. Primero suban `entrada.zip` (imágenes) y después sus videos (pueden elegir varios a la vez).
3. Al final se descarga `salida.zip`. Pónganlo en `Pipeline/colab/` y avísenle a Claude.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Instalar LivePortrait y descargar sus pesos
%cd /content
!rm -rf LivePortrait && git clone -q https://github.com/KwaiVGI/LivePortrait.git
%cd /content/LivePortrait
!pip -q install tyro onnxruntime-gpu pykalman lmdb imageio-ffmpeg 'huggingface_hub[cli]' rich
!pip -q install -r requirements_base.txt --no-deps 2>/dev/null; true
!huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights --exclude '*.git*' 'README.md' 'docs' > /dev/null
!ls pretrained_weights

In [ ]:
# Subir entrada.zip
from google.colab import files
import zipfile, os, shutil
shutil.rmtree('/content/entrada', ignore_errors=True)
up = files.upload()
for n in up: zipfile.ZipFile(n).extractall('/content/entrada')
print(sorted(os.listdir('/content/entrada')))

In [ ]:
# 4. Subir los videos de actuación (1.20.mp4, 1.07.mp4…)
os.makedirs('/content/actuacion', exist_ok=True)
for n, b in files.upload().items(): open('/content/actuacion/' + n, 'wb').write(b)
print(sorted(os.listdir('/content/actuacion')))

In [ ]:
# 5. Transferir la actuación a cada personaje
import glob, subprocess, shutil
os.makedirs('/content/salida', exist_ok=True)
for vid in sorted(glob.glob('/content/actuacion/*.mp4')):
    fila = os.path.basename(vid)[:-4]; img = f'/content/entrada/{fila}.png'
    if not os.path.exists(img): print('no hay imagen para', fila); continue
    out = f'/content/tmp/{fila}'; shutil.rmtree(out, ignore_errors=True)
    print('animando', fila)
    r = subprocess.run(['python', 'inference.py', '-s', img, '-d', vid, '-o', out, '--flag_crop_driving_video'], capture_output=True, text=True)
    vids = [v for v in glob.glob(out + '/*.mp4') if 'concat' not in v]
    if vids:
        shutil.copy(vids[0], f'/content/salida/{fila}.mp4')
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', vid, '-vn', '-ac', '2', '-ar', '48000', f'/content/salida/{fila}-voz.wav'])
        print('  listo')
    else: print('  ERROR\n', r.stdout[-1500:], r.stderr[-3000:])

In [ ]:
# Descargar salida.zip
!cd /content/salida && zip -q -r /content/salida.zip . && ls -la /content/salida
from google.colab import files
files.download('/content/salida.zip')